# diagnostic.ipynb

**Backend notebook for mī lyte System 1 diagnostics.**

This notebook provides:
1. **FAISS Index Building** - PDF loading, chunking, and vectorization
2. **Query Testing** - RAG queries with exposed reasoning traces and source citations

Use this notebook to:
- Rebuild the vector index when the knowledge base changes
- Inspect model reasoning during response generation
- Verify retrieved document relevance
- Debug and tune RAG pipeline behavior

---

> `diagnostic.ipynb`  
> Simone J. Skeen x Claude Code (07-11-2026)  
> WIP - NOT FOR DISTRIBUTION

## 1. Setup

Install dependencies, configure environment, import modules.

---

**Prerequisites:**
- Install dependencies: `pip install -r requirements.txt` from project root
- Pull embedding model: `ollama pull nomic-embed-text`
- Pull LLM model: `ollama pull deepseek-r1:14b`
- Create `.env` file with `KNOW_DIR` path to knowledge base PDFs

In [ ]:
%%capture

# === INSTALL DEPENDENCIES === #
# Capture output to keep notebook clean.
# Run this cell once per environment setup.

%pip install -r ../requirements.txt

In [ ]:
# === STANDARD LIBRARY IMPORTS === #

import os
import sys
import warnings
from pathlib import Path

# === THIRD-PARTY IMPORTS === #

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# LangChain ecosystem
import langchain
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.llms import Ollama
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# IPython display configuration
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# === PANDAS DISPLAY OPTIONS === #
# Show all columns and rows for debugging.

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# === SUPPRESS WARNINGS === #
# Hide FutureWarning and UserWarning to keep output clean.

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

# === LOAD ENVIRONMENT VARIABLES === #
# The .env file should contain KNOW_DIR path to knowledge base PDFs.

load_dotenv(Path('..') / '.env')

print(f"LangChain version: {langchain.__version__}")

In [ ]:
# === ADD PROJECT ROOT TO PATH === #
# This allows importing from src/ modules.
# The notebook runs from prototype/, so we add the parent directory.

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path().resolve()}")

In [ ]:
# === IMPORT FROM SRC MODULES === #
# - dialogue_stream.py: Unified streaming function with mode parameter
# - config.py: LLM parameters, retriever settings, prompt template

from src.dialogue_stream import query_and_stream  # type: ignore
from src.config import (  # type: ignore
    SYSTEM_PROMPT,
    LLM_PARAMS,
    EMBEDDING_MODEL,
    RETRIEVER_PARAMS,
    PROMPT_TEMPLATE,
)

print("Imports successful.")

## 2. Build FAISS Index

Load PDFs from knowledge base, chunk documents, create embeddings, and save vector index.

---

**When to rebuild:**
- Knowledge base PDFs have been updated
- Chunking parameters need adjustment
- Embedding model has changed

**Skip this section** if index already exists and knowledge base hasn't changed.

### 2a. Load PDFs

In [ ]:
# === KNOWLEDGE BASE DIRECTORY === #
# Path loaded from .env file. Expand ~ to full user path.

KNOW_DIR = os.path.expanduser(os.environ['KNOW_DIR'])
print(f"Knowledge base directory: {KNOW_DIR}")

# === PDF PATHS === #
# List all knowledge base documents to include in the index.

pdf_paths = [
    os.path.join(KNOW_DIR, "mbqr_manual_rag_db.pdf"),
    os.path.join(KNOW_DIR, "mbqr_scripts_rag_db.pdf"),
    os.path.join(KNOW_DIR, "poems_of protest_resistance_empowerment_rag_db_prelim.pdf"),
]

# === LOAD DOCUMENTS === #
# PyMuPDFLoader extracts text from PDFs with better formatting than PyPDFLoader.
# Each PDF is loaded as a list of Document objects (one per page).

all_documents = []

for path in pdf_paths:
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    all_documents.append(docs)  # List of lists (grouped by file)
    print(f"{os.path.basename(path)}: {len(docs)} pages loaded")

print(f"\nTotal files loaded: {len(all_documents)}")

In [ ]:
# === SPOT CHECK: DOCUMENT CONTENT === #
# Display first 1000 characters of each file's first page.
# Verify that text extraction is working correctly.

for i, doc_pages in enumerate(all_documents):
    print(f"\n{'='*60}")
    print(f"KNOWLEDGE SOURCE {i+1}")
    print(f"{'='*60}\n")
    print(doc_pages[0].page_content[:1000])

### 2b. Chunk and Vectorize

In [ ]:
# === FLATTEN DOCUMENT LIST === #
# Convert list of lists into single flat list of Document objects.

flat_documents = [page for doc in all_documents for page in doc]
print(f"Total pages: {len(flat_documents)}")

# === CHUNK DOCUMENTS === #
# RecursiveCharacterTextSplitter provides intelligent text splitting:
# - chunk_size: Maximum characters per chunk (1000 = ~250 words)
# - chunk_overlap: Characters shared between adjacent chunks (200 = context continuity)
# - separators: Priority order for split points (paragraphs > lines > words > chars)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', ' ', ''],
)

chunked_documents = splitter.split_documents(flat_documents)
print(f"Chunks created: {len(chunked_documents)}")

# === CREATE EMBEDDINGS === #
# OllamaEmbeddings uses nomic-embed-text model for semantic vector representations.
# Each chunk is converted to a dense vector for similarity search.

embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

# === BUILD FAISS INDEX === #
# FAISS (Facebook AI Similarity Search) enables fast nearest-neighbor lookup.
# from_documents() embeds all chunks and builds the index.

print("Building FAISS index (this may take a minute)...")
db = FAISS.from_documents(chunked_documents, embedding)
print("FAISS index built successfully.")

# === SAVE INDEX === #
# Persist to disk for reuse by demo.py and future notebook sessions.

db.save_local("../src/faiss_index")
print("Index saved to: ../src/faiss_index")

## 3. Query and Inspect

Test RAG queries with full reasoning trace and source document visibility.

---

**Diagnostic mode features:**
- `<think>...</think>` reasoning traces printed visibly
- Retrieved document excerpts with metadata
- Real-time token streaming for debugging

### 3a. Load Existing Index

Skip this cell if you just built the index above.

In [ ]:
# === LOAD SAVED FAISS INDEX === #
# Use this cell if the index already exists and doesn't need rebuilding.
# allow_dangerous_deserialization is required for loading pickled index.

embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)

db = FAISS.load_local(
    "../src/faiss_index",
    embeddings=embedding,
    allow_dangerous_deserialization=True,
)

print("FAISS index loaded successfully.")

### 3b. Configure LLM and Retriever

In [ ]:
# === CONFIGURE LLM === #
# Uses Ollama with DeepSeek-R1 for local inference.
# Parameters imported from src/config.py.

llm = Ollama(**LLM_PARAMS)
print(f"LLM model: {LLM_PARAMS['model']}")

# === CONFIGURE RETRIEVER === #
# Retriever settings imported from src/config.py.
# Default: similarity search with k=4 documents.

retriever = db.as_retriever(**RETRIEVER_PARAMS)
print(f"Retriever: {RETRIEVER_PARAMS}")

# === PROMPT TEMPLATE === #
# Imported from config - includes system prompt with tone/style guidance.

prompt = PROMPT_TEMPLATE
print("\nSystem prompt:")
print(SYSTEM_PROMPT[:1500] + "...") ### display system prompt; configurable

### 3c. Define Test Query

In [ ]:
# === TEST QUERY === #
# Modify this query to test different scenarios.

query = """
I have too much to deal with today! I feel so overwhelmed I can't even start.
"""

# === ALTERNATIVE TEST QUERIES === #
# Uncomment any of these to test different scenarios:

# query = "I want to die"  # Tests guardrail handling
# query = "I sometimes struggle with negative feelings toward my body"
# query = "Please tell me an inspiring quote. The world feels like too much lately."

print(f"Query: {query.strip()}")

### 3d. Execute Query (Diagnostic Mode)

In [ ]:
# === EXECUTE RAG QUERY === #
# Use mode='diagnostic' to expose:
#   - Full reasoning trace (<think>...</think> tags)
#   - Retrieved document excerpts with metadata
#   - Real-time token streaming to stdout
#
# This is the same streaming function used by demo.py,
# but in diagnostic mode instead of ui mode.

query_and_stream(
    llm,
    retriever,
    query,
    prompt_template=prompt,
    mode='diagnostic',   # Exposes reasoning + prints sources
    show_sources=True,   # Print retrieved document excerpts
)

---

> End of `diagnostic.ipynb`